# FOLIO → TPTP (v9) + Vampire + Robustez (ruído irrelevante e remoção de premissa)

Este notebook faz:

1. Carregar o FOLIO via Hugging Face (`yale-nlp/FOLIO`)
2. Converter `premises-FOL` e `conclusion-FOL` para TPTP (parser **v9** normalizado)
3. Rodar o **Vampire** (portfolio `casc` → fallback `casc_sat`) para rotular `True/False/Uncertain`
4. Selecionar apenas exemplos **convergentes** (Vampire == gold) e reportar **cobertura** e **acurácia**
5. Para um exemplo convergente:
   - Gerar um **fato irrelevante** (ruído) e verificar que o rótulo não muda
   - Encontrar ao menos uma **premissa relevante** (via ablation) para remover e observar mudança de rótulo

> Observação: como o próprio dataset pode conter ruído/typos, trabalhar com cobertura + acurácia nos convergentes é uma escolha metodológica sólida.


In [10]:
# Instala/Imports
import os, re, subprocess, random, textwrap
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Set

import pandas as pd
from datasets import load_dataset

import hashlib, pickle
from tqdm.notebook import tqdm

In [ ]:
# Configurações
SPLIT = "validation"   # 'validation' ou 'train' para FOLIO; altere se necessário
N_SAMPLES = 203        # max: 203 em validation, 1001 em train
SEED = 7

TIME_LIMIT = 600       # segundos por check (entails/contra)
CASC_FRACTION = 0.67   # fração do tempo para 'casc' antes do fallback 'casc_sat'

# Caminho do Vampire:
# - Se você instalou via apt/brew: "vampire"
# - Se você baixou binário local: aponte para o caminho completo.
VAMPIRE_BIN = "/usr/local/bin/vampire"

OUTDIR = "out_v10"
os.makedirs(OUTDIR, exist_ok=True)

random.seed(SEED)


# Quantos fatos de ruído irrelevante adicionar (1..K)
K_NOISE = 1

# Cache do Vampire (pickle)
CACHE_PATH = os.path.join(OUTDIR, 'vampire_cache.pkl')

# Quantos exemplos convergentes usar na geração em lote (None = todos os convergentes do batch)
N_CONVERGENT_MAX = 10

# Salvar tabela de perturbações em CSV
SAVE_PERTURB_CSV = True
PERTURB_CSV_PATH = os.path.join(OUTDIR, 'perturbations_knoise.csv')

# --- Relevância por remoção (split inteiro) ---
RELEVANCE_TIME_LIMIT = TIME_LIMIT   # pode aumentar (ex.: 180/240) se quiser mais "relevant_found"
MAX_PREMISES_TO_TEST = None         # None = testa todas; ou um inteiro (ex.: 25) para limitar custo
RUN_RELEVANCE_FULL_SPLIT = True
RELEVANCE_OUTDIR = os.path.join(OUTDIR, "relevance_fullsplit")
RELEVANCE_CSV_PATH = os.path.join(RELEVANCE_OUTDIR, "relevant_premise_%s.csv"%(SPLIT))

In [12]:
# Cache do Vampire (em memória + em disco)
# O cache é indexado pelo hash do conteúdo do arquivo TPTP + time_limit + casc_fraction.
VAMPIRE_CACHE: Dict[Tuple[str,int,float], Dict[str, object]] = {}

def _sha1_text(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def load_cache(path: str = CACHE_PATH) -> None:
    global VAMPIRE_CACHE
    if os.path.exists(path):
        try:
            with open(path, "rb") as f:
                VAMPIRE_CACHE = pickle.load(f)
            print(f"[cache] carregado: {len(VAMPIRE_CACHE)} entradas")
        except Exception as e:
            print("[cache] falha ao carregar, iniciando vazio:", e)
            VAMPIRE_CACHE = {}
    else:
        print("[cache] arquivo inexistente, iniciando vazio")

def save_cache(path: str = CACHE_PATH) -> None:
    try:
        with open(path, "wb") as f:
            pickle.dump(VAMPIRE_CACHE, f)
    except Exception as e:
        print("[cache] falha ao salvar:", e)

load_cache()


[cache] arquivo inexistente, iniciando vazio


In [13]:
# Carregar dataset
ds_all = load_dataset("yale-nlp/FOLIO")
ds = ds_all[SPLIT]
len(ds), ds[0].keys()


(203,
 dict_keys(['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id']))

## Parser v9 (normalização + NUM + identifiers com `_`)

Normalizações aplicadas (conservadoras):

- `-` → `_`
- balanceamento simples de parênteses (adiciona `)` no fim se faltar)
- símbolos (predicados/constantes/funções) para lowercase no TPTP
- números e `y1984` → `num1984` (mesmo símbolo)
- tokenização aceita `NUM`

Variáveis seguem a convenção: **bound vars** viram `X, Y, ...` (maiúsculas).


In [14]:
# --- ASTs ---
@dataclass(frozen=True)
class Tok:
    kind: str
    val: str

class ParseError(Exception):
    pass

@dataclass(frozen=True)
class Term: ...
@dataclass(frozen=True)
class TConst(Term):
    name: str
@dataclass(frozen=True)
class TFun(Term):
    name: str
    args: List[Term]

@dataclass(frozen=True)
class Fml: ...
@dataclass(frozen=True)
class Atom(Fml):
    pred: str
    args: List[Term]
@dataclass(frozen=True)
class Not(Fml):
    x: Fml
@dataclass(frozen=True)
class Bin(Fml):
    op: str  # AND OR XOR IMPL IFF
    a: Fml
    b: Fml
@dataclass(frozen=True)
class Quant(Fml):
    q: str   # FORALL EXISTS
    vars: List[str]
    body: Fml
@dataclass(frozen=True)
class Eq(Fml):
    op: str  # EQ NEQ
    a: Term
    b: Term

# --- Normalização / tokenizer ---
def normalize_line_v9(s: str) -> str:
    """
    Normalização conservadora para aumentar cobertura do parser no FOLIO.

    Nota: as regexes aqui usam escapes padrão (\b, \s, \(), etc).
    """
    import unicodedata
    if s is None:
        return ""
    s = s.strip()

    # Remove pontuação final comum (ex.: ponto final) que não faz parte da lógica
    s = re.sub(r"[.]+\s*$", "", s)

    # Remove comentários/explicações em linguagem natural
    if " Never:" in s:
        s = s.split(" Never:")[0].strip()

    # Unicode -> ascii-ish (remove acentos/diacríticos)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    # Normaliza flechas e bicondicionais
    s = s.replace("—>", "→").replace("–>", "→").replace("->", "→")
    s = s.replace("⟷", "↔").replace("⇔", "↔").replace("⟺", "↔")

    # Normaliza OR escrito como " v " em alguns exemplos
    s = re.sub(r"(?<=\s)v(?=\s)", "∨", s)

    # Preserva decimais como placeholders, para não destruir o ponto quando trocarmos '.' por '_'
    decimals = {}
    def _dec_repl(m):
        key = f"__DEC{len(decimals)}__"
        decimals[key] = m.group(0)
        return key
    s = re.sub(r"\b\d+\.\d+\b", _dec_repl, s)

    # Normaliza caracteres em identificadores
    s = s.replace("-", "_")
    s = s.replace("'", "_")

    # Pontos: substitui por '_' (não temos token '.'), depois restaura decimais
    s = s.replace(".", "_")

    # Restaura decimais
    for k, v in decimals.items():
        s = s.replace(k, v)

    # Compacta espaços
    s = re.sub(r"\s+", " ", s)

    # Remove vírgulas sobrando antes de ')'
    s = s.replace(",)", ")").replace(",,", ",")

    # Heurística: faltou um ')'
    # In(lily, jameSFamily ∧ WatchIn(...  -> In(lily, jameSFamily) ∧ WatchIn(...)
    s = re.sub(r"\bIn\(\s*([A-Za-z0-9_]+)\s*,\s*([A-Za-z0-9_]+)\s*∧", r"In(\1, \2) ∧", s)

    # Heurística: padrões como Pred((x) -> Pred(x)
    s = re.sub(r"([A-Za-z_][A-Za-z0-9_]*)\(\(\s*([A-Za-z_][A-Za-z0-9_]*)\s*\)", r"\1(\2)", s)

    # Heurística: quantificador sem variável: ∃( ... ) -> ∃x ( ... )
    s = re.sub(r"∃\s*\(", "∃x (", s)
    s = re.sub(r"∀\s*\(", "∀x (", s)

    # Heurística: argumentos separados por espaço / argumento com espaço dentro de lista com vírgulas
    def _maybe_commify_args(m):
        inner = m.group(1)

        # Se tiver conectivos/quantificadores/igualdade, não mexe.
        if any(sym in inner for sym in ["∀","∃","¬","∧","∨","⊕","→","↔","=","≠"]):
            return "(" + inner + ")"

        # Caso 1: não há vírgulas e há espaços -> transformar espaços em separadores de argumentos
        if "," not in inner and " " in inner:
            inner2 = re.sub(r"\s+", ", ", inner.strip())
            return "(" + inner2 + ")"

        # Caso 2: já há vírgulas, mas algum 'argumento' contém espaço (ex.: billieEilish BadGuy)
        if "," in inner:
            parts = [p.strip() for p in inner.split(",")]
            new_parts = []
            for p in parts:
                if " " in p:
                    toks = [t for t in re.split(r"\s+", p) if t]
                    if all(re.fullmatch(r"[A-Za-z0-9_]+|\d+\.\d+|\d+", t) for t in toks):
                        new_parts.extend(toks)
                    else:
                        new_parts.append(p)
                else:
                    new_parts.append(p)
            return "(" + ", ".join(new_parts) + ")"

        return "(" + inner + ")"

    s = re.sub(r"\(([^()]*)\)", _maybe_commify_args, s)

    # Remove ')' que não têm '(' correspondente (scan)
    out_chars = []
    bal = 0
    for ch in s:
        if ch == "(":
            bal += 1
            out_chars.append(ch)
        elif ch == ")":
            if bal > 0:
                bal -= 1
                out_chars.append(ch)
            else:
                continue
        else:
            out_chars.append(ch)
    s = "".join(out_chars)

    # Fecha '(' remanescentes
    if bal > 0:
        s = s + (")" * bal)

    return s
TOK_RE_V9 = re.compile(
    r"\s*(?:"
    r"(?P<FORALL>∀)|"
    r"(?P<EXISTS>∃)|"
    r"(?P<NOT>¬)|"
    r"(?P<AND>∧)|"
    r"(?P<OR>∨)|"
    r"(?P<XOR>⊕)|"
    r"(?P<IMPL>→)|"
    r"(?P<IFF>↔)|"
    r"(?P<EQ>=)|"
    r"(?P<NEQ>≠)|"
    r"(?P<LP>\()|"
    r"(?P<RP>\))|"
    r"(?P<COMMA>,)|"
    # ID: normal ou começa com dígitos e depois letras (ex.: 2008SummerOlympics)
    r"(?P<ID>(?:[A-Za-z_][A-Za-z0-9_]*|\d+[A-Za-z_][A-Za-z0-9_]*))|"
    # NUM: inteiro ou decimal
    r"(?P<NUM>\d+(?:\.\d+)?)"
    r")"
)

def tokenize_v9(s: str) -> List[Tok]:
    s = normalize_line_v9(s)
    out: List[Tok] = []
    i = 0
    n = len(s)
    while i < n:
        m = TOK_RE_V9.match(s, i)
        if not m:
            i += 1
            continue
        kind = None
        val = m.group(0).strip()
        for k, v in m.groupdict().items():
            if v is not None:
                kind = k
                val = v
                break
        out.append(Tok(kind, val))
        i = m.end()
    out.append(Tok("EOF", ""))
    return out

# --- Parser (precedência padrão: NOT > AND > OR > XOR > IMPL > IFF) ---
class Parser:
    def __init__(self, toks: List[Tok]):
        self.toks = toks
        self.i = 0

    def peek(self) -> Tok:
        return self.toks[self.i]

    def eat(self, kind: str) -> Tok:
        t = self.peek()
        if t.kind != kind:
            raise ParseError(f"Expected {kind}, got {t}")
        self.i += 1
        return t

    def accept(self, kind: str) -> Optional[Tok]:
        if self.peek().kind == kind:
            self.i += 1
            return self.toks[self.i - 1]
        return None

    def parse_fml(self) -> Fml:
        f = self.parse_iff()
        self.eat("EOF")
        return f

    def parse_iff(self) -> Fml:
        x = self.parse_impl()
        while self.accept("IFF"):
            y = self.parse_impl()
            x = Bin("IFF", x, y)
        return x

    def parse_impl(self) -> Fml:
        x = self.parse_xor()
        while self.accept("IMPL"):
            y = self.parse_xor()
            x = Bin("IMPL", x, y)
        return x

    def parse_xor(self) -> Fml:
        x = self.parse_or()
        while self.accept("XOR"):
            y = self.parse_or()
            x = Bin("XOR", x, y)
        return x

    def parse_or(self) -> Fml:
        x = self.parse_and()
        while self.accept("OR"):
            y = self.parse_and()
            x = Bin("OR", x, y)
        return x

    def parse_and(self) -> Fml:
        x = self.parse_not()
        while self.accept("AND"):
            y = self.parse_not()
            x = Bin("AND", x, y)
        return x

    def parse_not(self) -> Fml:
        if self.accept("NOT"):
            return Not(self.parse_not())
        return self.parse_atomlike()

    def parse_atomlike(self) -> Fml:
        if self.accept("LP"):
            f = self.parse_iff()
            self.eat("RP")
            return f
        if self.peek().kind in ("FORALL", "EXISTS"):
            # Suporta quantificadores em cadeia, ex.: ∀x ∀y ( ... ) e também mistos ∃x ∀y ( ... )
            segments = []
            while self.peek().kind in ("FORALL", "EXISTS"):
                qtok = self.eat(self.peek().kind)
                qkind = qtok.kind
                # Variáveis do quantificador.
                # IMPORTANTE: não consome IDs arbitrários (p.ex. nome de predicado) como variável.
                # Heurística: variáveis tendem a ser minúsculas (x,y,z,...) enquanto predicados começam com maiúscula.
                var_pat = re.compile(r"^[a-z][a-z0-9_]*$")
                vars_ = [self.eat("ID").val]
                # aceita lista curta de variáveis (ex.: ∀x y ( ... )) enquanto continuarem parecendo variáveis
                while self.peek().kind == "ID" and var_pat.fullmatch(self.peek().val or ""):
                    vars_.append(self.eat("ID").val)
                # aceita também vírgulas (ex.: ∀x, y ( ... ))
                while self.accept("COMMA"):
                    v = self.eat("ID").val
                    vars_.append(v)
                if not vars_:
                    raise ParseError(f"Quantifier without variables near {qtok}")
                segments.append((qkind, vars_))

            # Corpo: preferencialmente entre parênteses
            if self.accept("LP"):
                body = self.parse_iff()
                self.eat("RP")
            else:
                # fallback: corpo sem parênteses
                body = self.parse_atomlike()

            f = body
            for qkind, vars_ in reversed(segments):
                q = "FORALL" if qkind == "FORALL" else "EXISTS"
                f = Quant(q, vars_, f)
            return f

        # igualdade/diferença ou predicado
        # parse_term (primeiro termo)
        t1 = self.parse_term()
        # se vier EQ/NEQ, é fórmula de igualdade
        if self.peek().kind in ("EQ","NEQ"):
            op = "EQ" if self.accept("EQ") else "NEQ"
            t2 = self.parse_term()
            return Eq(op, t1, t2)

        # caso contrário, t1 deve ser um "predicado" sem args (tratamos como Atom 0-ária)
        if isinstance(t1, TConst):
            # se próximo token é LP, interpretamos como predicado com args (função "aplicada" como atom)
            if self.accept("LP"):
                args: List[Term] = []
                if not self.accept("RP"):
                    while True:
                        args.append(self.parse_term())
                        if self.accept("COMMA"):
                            continue
                        self.eat("RP")
                        break
                return Atom(t1.name, args)
            return Atom(t1.name, [])
        if isinstance(t1, TFun):
            # se parseou como função f(args) aqui, tratamos como predicado f(args)
            return Atom(t1.name, t1.args)

        raise ParseError("Unexpected atomlike")

    def parse_term(self) -> Term:
        idt = self.accept("ID")
        if idt:
            name = idt.val
            if self.accept("LP"):
                args: List[Term] = []
                if not self.accept("RP"):
                    while True:
                        args.append(self.parse_term())
                        if self.accept("COMMA"):
                            continue
                        self.eat("RP")
                        break
                return TFun(name, args)
            return TConst(name)

        numt = self.accept("NUM")
        if numt:
            return TConst(numt.val)

        raise ParseError(f"Expected term, got {self.peek()}")

def parse_folio_formula_v9(s: str) -> Fml:
    return Parser(tokenize_v9(s)).parse_fml()

# --- TPTP symbol canonicalization ---
NUM_LIKE = re.compile(r"^(?:y)?(\d+)$", re.IGNORECASE)

def canon_sym_v9(name: str, *, is_var: bool) -> str:
    if is_var:
        return name[:1].upper() + name[1:]
    m = NUM_LIKE.match(name)
    if m:
        return "num" + m.group(1)
    name2 = name.lower()
    name2 = re.sub(r"[^a-z0-9_]", "_", name2)
    name2 = re.sub(r"_+", "_", name2).strip("_")
    if not name2:
        name2 = "c"
    if name2[0].isdigit():
        name2 = "c_" + name2
    return name2

def term_to_tptp_v9(t: Term, bound: Set[str]) -> str:
    if isinstance(t, TConst):
        return canon_sym_v9(t.name, is_var=(t.name in bound))
    if isinstance(t, TFun):
        fname = canon_sym_v9(t.name, is_var=False)
        return f"{fname}({', '.join(term_to_tptp_v9(a, bound) for a in t.args)})"
    raise TypeError(t)

def fml_to_tptp_v9(f: Fml, bound: Optional[Set[str]] = None) -> str:
    bound = set(bound or set())
    if isinstance(f, Quant):
        vars_tptp = [canon_sym_v9(v, is_var=True) for v in f.vars]
        new_bound = set(bound) | set(f.vars)
        qsym = "!" if f.q == "FORALL" else "?"
        return f"{qsym} [{', '.join(vars_tptp)}] : ({fml_to_tptp_v9(f.body, new_bound)})"
    if isinstance(f, Atom):
        pred = canon_sym_v9(f.pred, is_var=False)
        if not f.args:
            return pred
        return f"{pred}({', '.join(term_to_tptp_v9(a, bound) for a in f.args)})"
    if isinstance(f, Not):
        return f"~({fml_to_tptp_v9(f.x, bound)})"
    if isinstance(f, Bin):
        op_map = {"AND":"&","OR":"|","IMPL":"=>","IFF":"<=>","XOR":"<~>"}
        op = op_map[f.op]
        return f"({fml_to_tptp_v9(f.a, bound)} {op} {fml_to_tptp_v9(f.b, bound)})"
    if isinstance(f, Eq):
        op = "=" if f.op == "EQ" else "!="
        return f"({term_to_tptp_v9(f.a, bound)} {op} {term_to_tptp_v9(f.b, bound)})"
    raise TypeError(f)

def folio_to_tptp_v9(formula: str) -> str:
    formula = normalize_line_v9(formula)
    return fml_to_tptp_v9(parse_folio_formula_v9(formula))

def safe_folio_to_tptp_v9(formula: str) -> Tuple[Optional[str], Optional[str]]:
    try:
        return folio_to_tptp_v9(formula), None
    except Exception as e:
        return None, str(e)


<>:50: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:50: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_46163/27713222.py:50: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  Nota: as regexes aqui usam escapes padrão (\b, \s, \(), etc).


## Diagnóstico rápido: cobertura do parser (v9.2) no split inteiro

In [15]:
# Checa quantos conclusions/premises parseiam no split inteiro (rápido, sem Vampire)
def parse_coverage(ds_split):
    n = len(ds_split)
    bad = []
    for i, ex in enumerate(tqdm(ds_split)):
        premises_fol = ex.get("premises-FOL") or ex.get("premises_fol") or ""
        conclusion_fol = ex.get("conclusion-FOL") or ex.get("conclusion_fol") or ""
        concl_tptp, concl_err = safe_folio_to_tptp_v9(str(conclusion_fol).strip())
        if concl_err or concl_tptp is None:
            bad.append((i, ex.get("example_id", i), "conclusion", concl_err, conclusion_fol))
            continue
        for ln in [l.strip() for l in str(premises_fol).splitlines() if l.strip()]:
            t, e = safe_folio_to_tptp_v9(ln)
            if e or t is None:
                bad.append((i, ex.get("example_id", i), "premise", e, ln))
                break
    return n, bad

n_total, bad_list = parse_coverage(ds)
print("Total:", n_total)
print("Parse OK:", n_total - len(bad_list))
print("Parse ERR:", len(bad_list))
pd.DataFrame(bad_list, columns=["dataset_idx","example_id","where","error","detail"]).head(30)


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## Runner do Vampire (portfolio `casc` → `casc_sat`) + rotulador

Critério:

- `True`  se `premises ∧ ¬conclusion` é **UNSAT**
- `False` se `premises ∧ conclusion` é **UNSAT**
- caso contrário `Uncertain`


In [ ]:
SZS_RE = re.compile(r"^\s*%?\s*SZS\s+status\s+(\w+)", re.IGNORECASE)
DECISIVE_SZS = {"unsatisfiable","satisfiable","theorem","countersatisfiable","contradictory"}

def _parse_szs(output: str) -> str:
    for line in output.splitlines():
        m = SZS_RE.search(line)
        if m:
            return m.group(1)
    return "Unknown"

def run_vampire_once(tptp_file: str, time_limit: int, schedule: str) -> Tuple[str, str]:
    cmd = [
        VAMPIRE_BIN,
        "--mode", "portfolio",
        "--schedule", schedule,
        "-t", str(time_limit),
        "--input_syntax", "tptp",
        "--output_mode", "szs",
        tptp_file,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    out = (proc.stdout or "") + "\n" + (proc.stderr or "")
    return _parse_szs(out), out

def run_vampire(tptp_file: str, time_limit: int = TIME_LIMIT, casc_fraction: float = CASC_FRACTION) -> Tuple[str, str, Dict[str, object]]:
    t1 = max(1, int(time_limit * casc_fraction))
    t2 = max(1, time_limit - t1)
    szs1, out1 = run_vampire_once(tptp_file, t1, "casc")
    if szs1.lower() in DECISIVE_SZS:
        return szs1, out1, {"used_schedule":"casc","t_casc":t1,"t_casc_sat":t2,"szs_casc":szs1}
    szs2, out2 = run_vampire_once(tptp_file, t2, "casc_sat")
    return szs2, out2, {"used_schedule":"casc_sat","t_casc":t1,"t_casc_sat":t2,"szs_casc":szs1,"szs_casc_sat":szs2}

def is_unsat(szs: str) -> bool:
    return szs.lower() in {"unsatisfiable","contradictory"}

def write_tptp_problem_v9(premises_lines: List[str], extra_axioms_tptp: List[str], out_path: str, comment: str = "") -> Dict[str, object]:
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    prem_tptp = []
    prem_errors = []
    for p in premises_lines:
        tptp, err = safe_folio_to_tptp_v9(p)
        if err or tptp is None:
            prem_errors.append((p, err))
        else:
            prem_tptp.append(tptp)

    with open(out_path, "w", encoding="utf-8") as fp:
        if comment:
            fp.write(f"% {comment}\n")
        for i, tptp in enumerate(prem_tptp, start=1):
            fp.write(f"fof(p{i}, axiom, {tptp}).\n")
        if prem_errors:
            fp.write(f"% [WARN] {len(prem_errors)} premise lines failed to parse and were skipped.\n")
            for (orig, err) in prem_errors[:10]:
                fp.write(f"%   line: {orig}\n")
                fp.write(f"%   err : {err}\n")
        for k, ax in enumerate(extra_axioms_tptp, start=1):
            fp.write(f"fof(extra{k}, axiom, {ax}).\n")

    return {"n_premises_skipped": len(prem_errors)}

def vampire_label_v9(premises_lines: List[str], conclusion_fol: str, out_prefix: str, time_limit: int = TIME_LIMIT) -> Dict[str, object]:
    concl_tptp, concl_err = safe_folio_to_tptp_v9(conclusion_fol.strip())
    if concl_err or concl_tptp is None:
        return {"vampire_label":"PARSE_ERROR","parse_error":f"Conclusion: {concl_err}"}

    entails_file = out_prefix + "_entails.p"
    meta_e = write_tptp_problem_v9(
        premises_lines, [f"~({concl_tptp})"], entails_file,
        comment="premises + ~conclusion (UNSAT => entails)"
    )
    szs_e, out_e, runmeta_e = run_vampire_cached(entails_file, time_limit=time_limit)

    contra_file = out_prefix + "_contra.p"
    meta_c = write_tptp_problem_v9(
        premises_lines, [f"({concl_tptp})"], contra_file,
        comment="premises + conclusion (UNSAT => contradiction)"
    )
    szs_c, out_c, runmeta_c = run_vampire_cached(contra_file, time_limit=time_limit)

    if is_unsat(szs_e):
        lab = "True"
    elif is_unsat(szs_c):
        lab = "False"
    else:
        lab = "Uncertain"

    return {
        "vampire_label": lab,
        "szs_entails": szs_e,
        "szs_contra": szs_c,
        "used_schedule_entails": runmeta_e.get("used_schedule"),
        "used_schedule_contra": runmeta_c.get("used_schedule"),
        "n_premises_skipped_entails": meta_e.get("n_premises_skipped"),
        "n_premises_skipped_contra": meta_c.get("n_premises_skipped"),
        "tptp_entails_file": entails_file,
        "tptp_contra_file": contra_file,
        "out_entails": out_e,
        "out_contra": out_c,
        "parse_error": None,
    }


def run_vampire_cached(tptp_file: str, time_limit: int = TIME_LIMIT, casc_fraction: float = CASC_FRACTION) -> Tuple[str, str, Dict[str, object]]:
    # Hash do conteúdo do arquivo (não do path), para cachear mesmo se o nome mudar.
    with open(tptp_file, "r", encoding="utf-8") as f:
        content = f.read()
    key = (_sha1_text(content), int(time_limit), float(casc_fraction))
    hit = VAMPIRE_CACHE.get(key)
    if hit is not None:
        return hit["szs"], hit["out"], hit["meta"]

    szs, out, meta = run_vampire(tptp_file, time_limit=time_limit, casc_fraction=casc_fraction)
    VAMPIRE_CACHE[key] = {"szs": szs, "out": out, "meta": meta}
    # salva de forma incremental para não perder em travamentos
    save_cache()
    return szs, out, meta


## Avaliação em lote + seleção de convergentes

- **Cobertura**: % de exemplos amostrados que não deram `PARSE_ERROR` e produziram rótulo (True/False/Uncertain)
- **Acurácia (nos válidos)**: % de acerto entre os válidos
- **Convergentes**: exemplos válidos onde `vampire_label == gold_label`


In [ ]:
# Amostrar índices
if N_SAMPLES >= len(ds):
    indices = list(range(len(ds)))
else:
    indices = random.sample(range(len(ds)), k=min(N_SAMPLES, len(ds)))
    indices[:10], len(indices)

In [ ]:
rows = []
for k, idx in enumerate(tqdm(indices), start=1):
    ex = ds[idx]
    premises_fol = ex.get("premises-FOL") or ex.get("premises_fol") or ""
    conclusion_fol = ex.get("conclusion-FOL") or ex.get("conclusion_fol") or ""
    gold = ex.get("label")
    example_id = ex.get("example_id", idx)

    premises_lines = [ln.strip() for ln in str(premises_fol).splitlines() if ln.strip()]
    out_prefix = os.path.join(OUTDIR, f"ex{example_id}_{k}")

    res = vampire_label_v9(premises_lines, str(conclusion_fol), out_prefix, time_limit=TIME_LIMIT)

    rows.append({
        "sample_k": k,
        "dataset_idx": idx,
        "example_id": example_id,
        "gold_label": gold,
        "vampire_label": res.get("vampire_label"),
        "match": (gold == res.get("vampire_label")),
        "szs_entails": res.get("szs_entails"),
        "szs_contra": res.get("szs_contra"),
        "n_premises": len(premises_lines),
        "parse_error": res.get("parse_error"),
    })

df = pd.DataFrame(rows)
df


In [ ]:
valid = df[df["vampire_label"].isin(["True","False","Uncertain"])].copy()
coverage = len(valid) / len(df) if len(df) else 0.0
accuracy_valid = (valid["gold_label"] == valid["vampire_label"]).mean() if len(valid) else 0.0

convergent = valid[valid["match"] == True].copy()
acc_convergent = (convergent["gold_label"] == convergent["vampire_label"]).mean() if len(convergent) else 0.0

print("N amostras:", len(df))
print("Cobertura (válidos):", round(coverage, 3), f"({len(valid)}/{len(df)})")
print("Acurácia (válidos):", round(accuracy_valid, 3))
print("N convergentes:", len(convergent))
print("Acurácia (convergentes):", round(acc_convergent, 3))

convergent[["sample_k","dataset_idx","example_id","gold_label","vampire_label","n_premises"]].head(20)


## (1) Gerar um fato irrelevante (ruído) e checar invariância

Critério de irrelevância **por construção** para o provador:

- Usa **predicado novo** e **constante nova** que não aparecem no problema
- Adiciona apenas um **fato positivo** `noise_p(noise_c)`

Em FOL padrão, isso não conecta com nada e não deve mudar entailment/contradição.


In [ ]:
def extract_symbol_bag(text: str) -> Set[str]:
    # pega IDs e também números (viram numXXXX no TPTP)
    toks = re.findall(r"[A-Za-z_][A-Za-z0-9_]*|\d+", normalize_line_v9(text))
    bag = set()
    for t in toks:
        # espelha o canon_sym_v9 (aprox.): lowercase e números->num
        m = NUM_LIKE.match(t)
        if m:
            bag.add("num" + m.group(1))
        else:
            bag.add(re.sub(r"[^a-z0-9_]", "_", t.lower()))
    return bag

def generate_irrelevant_fact(premises_lines: List[str], conclusion_fol: str, seed: int = 0) -> Tuple[str, str]:
    rng = random.Random(seed)
    bag = set()
    for p in premises_lines:
        bag |= extract_symbol_bag(p)
    bag |= extract_symbol_bag(conclusion_fol)

    # gera símbolos que não colidem
    for _ in range(10_000):
        pid = rng.randint(1, 10**9)
        pred = f"noise_p_{pid}"
        const = f"noise_c_{pid}"
        pred_c = canon_sym_v9(pred, is_var=False)
        const_c = canon_sym_v9(const, is_var=False)
        if pred_c not in bag and const_c not in bag:
            fol_fact = f"{pred}({const})"
            tptp_fact = f"{pred_c}({const_c})"
            return fol_fact, tptp_fact
    raise RuntimeError("Falha ao gerar fato irrelevante sem colisão.")


In [ ]:
# Selecionar um exemplo convergente (preferir True/False para o próximo passo)
tf = convergent[convergent["gold_label"].isin(["True","False"])].copy()
if len(tf) == 0:
    # fallback: qualquer convergente
    pick = convergent.iloc[0].to_dict()
else:
    pick = tf.iloc[0].to_dict()

pick


In [ ]:
idx = int(pick["dataset_idx"])
ex = ds[idx]

premises_fol = ex.get("premises-FOL") or ex.get("premises_fol") or ""
conclusion_fol = ex.get("conclusion-FOL") or ex.get("conclusion_fol") or ""
gold = ex.get("label")
example_id = ex.get("example_id", idx)

premises_lines = [ln.strip() for ln in str(premises_fol).splitlines() if ln.strip()]

print("example_id:", example_id, "gold:", gold, "n_premises:", len(premises_lines))
print("conclusion-FOL:", conclusion_fol)


In [ ]:
# Rodar baseline
base_res = vampire_label_v9(premises_lines, str(conclusion_fol), os.path.join(OUTDIR, f"demo_ex{example_id}_base"), time_limit=TIME_LIMIT)
base_res["vampire_label"], base_res["szs_entails"], base_res["szs_contra"]


In [ ]:
# Gerar K ruídos irrelevantes e testar aumento de ruído
def generate_k_irrelevant_facts(premises_lines: List[str], conclusion_fol: str, k: int, seed: int = 0) -> List[str]:
    facts = []
    # Para evitar colisões entre os próprios ruídos, vamos "reservar" o bag incrementalmente
    cur_prem = list(premises_lines)
    for j in range(k):
        fol_noise, _ = generate_irrelevant_fact(cur_prem, conclusion_fol, seed=seed + j)
        facts.append(fol_noise)
        cur_prem.append(fol_noise)
    return facts

noises = generate_k_irrelevant_facts(premises_lines, str(conclusion_fol), k=K_NOISE, seed=SEED)
print("K_NOISE =", K_NOISE)
for j, n in enumerate(noises, start=1):
    print(f"[noise {j}] {n}")

labels = []
for k_add in range(1, K_NOISE + 1):
    prem_k = premises_lines + noises[:k_add]
    res_k = vampire_label_v9(prem_k, str(conclusion_fol), os.path.join(OUTDIR, f"demo_ex{example_id}_noise{k_add}"), time_limit=TIME_LIMIT)
    labels.append((k_add, res_k["vampire_label"], res_k.get("szs_entails"), res_k.get("szs_contra")))
labels


## (2) Encontrar uma premissa relevante (ablation)

Para exemplos `gold ∈ {True, False}`, procuramos uma premissa tal que ao removê-la:

- se gold=True: deixe de ser provado (`True → Uncertain`) ou mude rótulo
- se gold=False: deixe de ser refutado (`False → Uncertain`) ou mude rótulo

Isso nos dá uma **premissa candidata relevante** para seu teste de robustez da LLM/LRM.


In [ ]:
def find_relevant_premises(
    premises_lines: List[str],
    conclusion_fol: str,
    gold_label: str,
    out_prefix: str,
    time_limit: int = TIME_LIMIT,
    max_found: int = 3,
) -> List[Dict[str, object]]:
    base = vampire_label_v9(premises_lines, conclusion_fol, out_prefix + "_base", time_limit=time_limit)
    base_lab = base["vampire_label"]
    if base_lab not in {"True","False","Uncertain"}:
        return []

    found = []
    for i in range(len(premises_lines)):
        pruned = premises_lines[:i] + premises_lines[i+1:]
        res = vampire_label_v9(pruned, conclusion_fol, out_prefix + f"_drop{i+1}", time_limit=time_limit)
        lab = res.get("vampire_label")
        if lab not in {"True","False","Uncertain"}:
            continue

        changed = (lab != base_lab)
        # critério de relevância pragmático
        relevant = False
        if gold_label == "True" and base_lab == "True":
            relevant = (lab != "True")  # perdeu prova
        elif gold_label == "False" and base_lab == "False":
            relevant = (lab != "False") # perdeu refutação
        else:
            relevant = changed

        if relevant:
            found.append({
                "premise_index": i,
                "premise_text": premises_lines[i],
                "base_label": base_lab,
                "new_label": lab,
                "base_szs_entails": base.get("szs_entails"),
                "base_szs_contra": base.get("szs_contra"),
                "new_szs_entails": res.get("szs_entails"),
                "new_szs_contra": res.get("szs_contra"),
            })
            if len(found) >= max_found:
                break
    return found


In [ ]:
# Rodar ablation e pegar ao menos 1 premissa relevante (se existir)
relevant = find_relevant_premises(
    premises_lines=premises_lines,
    conclusion_fol=str(conclusion_fol),
    gold_label=str(gold),
    out_prefix=os.path.join(OUTDIR, f"demo_ex{example_id}_relevance"),
    time_limit=TIME_LIMIT,
    max_found=3
)
relevant


In [ ]:
# Se encontrou uma premissa relevante, demonstre o efeito removendo-a (em Vampire)
if relevant:
    i = relevant[0]["premise_index"]
    pruned = premises_lines[:i] + premises_lines[i+1:]
    pruned_res = vampire_label_v9(pruned, str(conclusion_fol), os.path.join(OUTDIR, f"demo_ex{example_id}_pruned"), time_limit=TIME_LIMIT)
    print("Removed premise idx:", i)
    print("Removed premise:", premises_lines[i])
    print("Baseline label:", base_res["vampire_label"], "→ After removal:", pruned_res["vampire_label"])
else:
    print("Nenhuma premissa 'relevante' encontrada sob este critério/tempo. Tente outro exemplo convergente ou aumente TIME_LIMIT.")


## Geração em lote: perturbações com K ruídos para cada exemplo convergente

Esta seção cria uma tabela (`df_perturb`) com, para cada exemplo convergente no batch:

- condição `base` (sem ruído)
- condições `noise_1 ... noise_K` (adicionando os primeiros k fatos irrelevantes)

Colunas principais:
- `example_id`, `dataset_idx`
- `gold_label`, `vampire_label`
- `condition` (`base`, `noise_1`, ...)
- `k_noise` (0 para base)
- `noise_facts` (lista dos fatos adicionados naquela condição)

> A execução usa o **cache** do Vampire, então re-runs são bem mais rápidas.


In [ ]:
def build_perturbations_table(
    ds_split,
    convergent_df: pd.DataFrame,
    k_noise: int = 1,
    n_max: Optional[int] = 10,
    seed: int = 0,
) -> pd.DataFrame:
    # Seleciona exemplos convergentes
    conv = convergent_df.copy()
    if n_max is not None and len(conv) > n_max:
        conv = conv.sample(n=n_max, random_state=seed)

    rows = []
    for _, row in conv.iterrows():
        idx = int(row["dataset_idx"])
        ex = ds_split[idx]
        premises_fol = ex.get("premises-FOL") or ex.get("premises_fol") or ""
        conclusion_fol = ex.get("conclusion-FOL") or ex.get("conclusion_fol") or ""
        gold = ex.get("label")
        example_id = ex.get("example_id", idx)

        premises_lines = [ln.strip() for ln in str(premises_fol).splitlines() if ln.strip()]

        # baseline label (reusa o que já calculamos no df, mas recalcular aqui é ok por causa do cache)
        base_res = vampire_label_v9(
            premises_lines,
            str(conclusion_fol),
            os.path.join(OUTDIR, f"pert_ex{example_id}_base"),
            time_limit=TIME_LIMIT
        )
        base_lab = base_res.get("vampire_label")

        rows.append({
            "example_id": example_id,
            "dataset_idx": idx,
            "gold_label": gold,
            "condition": "base",
            "k_noise": 0,
            "noise_facts": [],
            "vampire_label": base_lab,
            "match_gold": (gold == base_lab),
        })

        # Gera K fatos irrelevantes (fixo por exemplo)
        noises = generate_k_irrelevant_facts(premises_lines, str(conclusion_fol), k=k_noise, seed=seed + int(example_id) % 1000000)

        # condições incrementais
        for k_add in range(1, k_noise + 1):
            prem_k = premises_lines + noises[:k_add]
            res_k = vampire_label_v9(
                prem_k,
                str(conclusion_fol),
                os.path.join(OUTDIR, f"pert_ex{example_id}_noise{k_add}"),
                time_limit=TIME_LIMIT
            )
            lab = res_k.get("vampire_label")
            rows.append({
                "example_id": example_id,
                "dataset_idx": idx,
                "gold_label": gold,
                "condition": f"noise_{k_add}",
                "k_noise": k_add,
                "noise_facts": noises[:k_add],
                "vampire_label": lab,
                "match_gold": (gold == lab),
            })

    return pd.DataFrame(rows)

df_perturb = build_perturbations_table(
    ds_split=ds,
    convergent_df=convergent,
    k_noise=K_NOISE,
    n_max=N_CONVERGENT_MAX,
    seed=SEED,
)

df_perturb


In [ ]:
# Resumo: em quantos exemplos o rótulo do Vampire mudou ao adicionar ruído?
# (idealmente 0, já que é irrelevante por construção)
summ = (
    df_perturb.pivot_table(index=["example_id","gold_label"], columns="k_noise", values="vampire_label", aggfunc="first")
)
summ["changed_vs_base"] = summ.apply(lambda r: any(r[k] != r[0] for k in range(1, K_NOISE+1) if k in summ.columns), axis=1)
print("N exemplos:", len(summ))
print("Mudou vs base:", int(summ["changed_vs_base"].sum()))
summ.sort_values("changed_vs_base", ascending=False).head(20)


In [ ]:
# Salvar CSV (opcional)
if SAVE_PERTURB_CSV:
    df_perturb.to_csv(PERTURB_CSV_PATH, index=False)
    print("Salvo em:", PERTURB_CSV_PATH)


## Split inteiro: encontrar 1 premissa relevante por exemplo (quando possível)

Ablation: remove uma premissa por vez e re-roda Vampire.

Filtramos para:
- `gold_label ∈ {True, False}`
- `vampire_label_base ∈ {True, False}`
- convergência: `vampire_label_base == gold_label`

Então buscamos uma premissa cuja remoção **quebra** a prova/refutação (vira `Uncertain` ou muda o rótulo).

Saída:
- `df_rel`
- CSV em `RELEVANCE_CSV_PATH`

> Pode ser caro; cache ajuda muito. Para limitar custo, use `MAX_PREMISES_TO_TEST`.


In [ ]:
import time

os.makedirs(RELEVANCE_OUTDIR, exist_ok=True)

def find_one_relevant_premise_for_example(
    premises_lines: List[str],
    conclusion_fol: str,
    gold_label: str,
    example_id,
    dataset_idx: int,
    time_limit: int,
) -> Dict[str, object]:
    out_prefix_base = os.path.join(RELEVANCE_OUTDIR, f"ex{example_id}_idx{dataset_idx}_base")
    base = vampire_label_v9(premises_lines, conclusion_fol, out_prefix_base, time_limit=time_limit)
    base_lab = base.get("vampire_label")

    if base_lab not in {"True", "False"}:
        return {"status":"SKIP_BASE_NOT_DECIDED", "base_label": base_lab, "relevant_found": False}
    if gold_label not in {"True", "False"}:
        return {"status":"SKIP_GOLD_NOT_DECIDED", "base_label": base_lab, "relevant_found": False}
    if base_lab != gold_label:
        return {"status":"SKIP_NOT_CONVERGENT", "base_label": base_lab, "relevant_found": False}

    n = len(premises_lines)
    n_test = n if MAX_PREMISES_TO_TEST is None else min(n, int(MAX_PREMISES_TO_TEST))

    for i in range(n_test):
        pruned = premises_lines[:i] + premises_lines[i+1:]
        out_prefix_drop = os.path.join(RELEVANCE_OUTDIR, f"ex{example_id}_idx{dataset_idx}_drop{i+1}")
        res = vampire_label_v9(pruned, conclusion_fol, out_prefix_drop, time_limit=time_limit)
        lab = res.get("vampire_label")
        if lab not in {"True", "False", "Uncertain"}:
            continue

        if base_lab == "True" and lab != "True":
            return {
                "status":"OK_RELEVANT_FOUND",
                "base_label": base_lab,
                "new_label": lab,
                "relevant_found": True,
                "premise_index": i,
                "premise_text": premises_lines[i],
                "n_premises": n,
            }
        if base_lab == "False" and lab != "False":
            return {
                "status":"OK_RELEVANT_FOUND",
                "base_label": base_lab,
                "new_label": lab,
                "relevant_found": True,
                "premise_index": i,
                "premise_text": premises_lines[i],
                "n_premises": n,
            }

    return {
        "status":"NO_RELEVANT_FOUND_UNDER_BUDGET",
        "base_label": base_lab,
        "relevant_found": False,
        "n_premises": n,
    }

if RUN_RELEVANCE_FULL_SPLIT:
    t0 = time.time()
    rows = []
    for idx in tqdm(range(len(ds))):
        ex = ds[idx]
        example_id = ex.get("example_id", idx)
        gold = str(ex.get("label"))

        premises_fol = ex.get("premises-FOL") or ex.get("premises_fol") or ""
        conclusion_fol = ex.get("conclusion-FOL") or ex.get("conclusion_fol") or ""
        premises_lines = [ln.strip() for ln in str(premises_fol).splitlines() if ln.strip()]

        r = find_one_relevant_premise_for_example(
            premises_lines=premises_lines,
            conclusion_fol=str(conclusion_fol).strip(),
            gold_label=gold,
            example_id=example_id,
            dataset_idx=idx,
            time_limit=RELEVANCE_TIME_LIMIT,
        )

        rows.append({
            "dataset_idx": idx,
            "example_id": example_id,
            "gold_label": gold,
            "status": r.get("status"),
            "base_label": r.get("base_label"),
            "relevant_found": r.get("relevant_found"),
            "new_label": r.get("new_label"),
            "premise_index": r.get("premise_index"),
            "premise_text": r.get("premise_text"),
            "n_premises": r.get("n_premises", len(premises_lines)),
        })

    df_rel = pd.DataFrame(rows)
    elapsed = time.time() - t0
    print("Done. Seconds:", round(elapsed, 1))
    print(df_rel["status"].value_counts())

    mask_conv_tf = df_rel["status"].isin(["OK_RELEVANT_FOUND","NO_RELEVANT_FOUND_UNDER_BUDGET"])
    n_conv_tf = int(mask_conv_tf.sum())
    n_found = int((df_rel["status"] == "OK_RELEVANT_FOUND").sum())
    print("Convergentes (gold True/False & base True/False):", n_conv_tf)
    print("Premissa relevante encontrada:", n_found, f"({(n_found/n_conv_tf if n_conv_tf else 0):.3f})")

    df_rel.to_csv(RELEVANCE_CSV_PATH, index=False)
    print("Saved:", RELEVANCE_CSV_PATH)

    df_rel.head(20)
else:
    print("RUN_RELEVANCE_FULL_SPLIT=False (pulei).")


RESULTADO

PARA os 203 exemplos da validação:

* 112: SKIP_BASE_NOT_DECIDED → o Vampire na base ficou Uncertain (ou seja, não dá pra fazer teste de “remoção de premissa relevante” porque não havia prova/refutação para “quebrar”).
* __86__: OK_RELEVANT_FOUND → aqui o Vampire provou/refutou e bateu com o gold, e sempre existe pelo menos uma premissa cuja remoção destrói o resultado (dentro do critério).
* 4: SKIP_NOT_CONVERGENT → Vampire decidiu True/False, mas discordou do gold.
* 1: SKIP_GOLD_NOT_DECIDED → gold é Uncertain (o script não entra na lógica de relevância por remoção).

PARA os 1001 exemplos do treino:

* 597: SKIP_BASE_NOT_DECIDED             
* __360__: OK_RELEVANT_FOUND                 
* 22: SKIP_NOT_CONVERGENT                
* 16: SKIP_GOLD_NOT_DECIDED              
* 6: NO_RELEVANT_FOUND_UNDER_BUDGET → estourou timeout antes de encontrar uma solução     

In [ ]:
!cat out_v10/relevance_fullsplit/relevant_premise_validation.csv | grep OK_RELEVANT_FOUND | head -10

In [ ]:
!cat out_v10/relevance_fullsplit/relevant_premise_train.csv | grep OK_RELEVANT_FOUND | head -10

## Próximos passos para seu experimento com LLM/LRM

Para cada exemplo convergente:

1. **Ruído irrelevante**: adicione `fol_noise` como linha extra em `premises` (ou uma frase NL equivalente) e veja se a LLM/LRM mantém o rótulo.
2. **Remoção relevante**: remova a premissa indicada por `find_relevant_premises` e veja se a LLM/LRM muda o rótulo (idealmente: perde certeza → `Uncertain`).

Dicas práticas:
- Para estabilidade, guarde também o arquivo `.p` TPTP usado em cada condição (já fica em `OUTDIR`).
- Faça caching dos resultados do Vampire para acelerar loops grandes.
